# Phase 3 (Mark): Domain Fragments + Feature Selection

**Date:** 2026-04-08 · **Researcher:** Mark Rodrigues · **Dataset:** ogbg-molhiv (scaffold split)

### Building on Anthony's Phase 3
Anthony's Phase 3 landed two big results:
1. **GIN + edge features = 0.7860 AUC** — first GNN to beat CatBoost (+0.008 over CatBoost+all-trad)
2. **Counterintuitive:** appending GNN embeddings to CatBoost+all-traditional (1345 dims) *hurts* by −0.037 vs 1217 dims. More features ≠ better.

He did NOT try:
- **RDKit Fragment (`Fr_*`) descriptors** — 85 explicit functional-group counts (aliphatic OH, aromatic amines, nitro, sulfonamide, halides, etc.) that medicinal chemists use for toxicophore/pharmacophore screening.
- **Feature selection to rescue the full hybrid** — is the 1345-d failure because of *useless noise*, or is CatBoost actually learning something the pruning would destroy?

### Hypotheses
- **H1 (domain knowledge):** 85 functional-group counts (Fr_*) alone beat Lipinski-14 (0.7744) because HIV inhibition depends on specific reactive groups.
- **H2 (curation):** Mutual-info top-K on Anthony's 1217-d all-traditional has a sweet spot that beats using all 1217.
- **H3 (headline):** `Lipinski + Fragments + top-K Morgan bits` CatBoost > Anthony's GIN+Edge champion 0.7860.


In [1]:
import os, json, time, warnings
warnings.filterwarnings('ignore')
os.environ['MPLBACKEND'] = 'Agg'
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Patch torch.load for OGB cache
import torch
_orig_load = torch.load
torch.load = lambda *a, **k: _orig_load(*a, **{**k, 'weights_only': False})

from ogb.graphproppred import GraphPropPredDataset
from rdkit import Chem
from rdkit.Chem import Fragments, Descriptors, rdMolDescriptors, AllChem, MACCSkeys
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.feature_selection import mutual_info_classif
from catboost import CatBoostClassifier

BASE = Path.cwd()
if BASE.name != 'Drug-Molecule-Property-Prediction':
    BASE = BASE.parent
RES = BASE / 'results'; RES.mkdir(exist_ok=True)

np.random.seed(42)
print('setup OK, base=', BASE.name)

setup OK, base= Drug-Molecule-Property-Prediction


In [2]:
t0 = time.time()
dataset = GraphPropPredDataset(name='ogbg-molhiv', root=str(BASE / 'data' / 'raw'))
split_idx = dataset.get_idx_split()
smiles_df = pd.read_csv(BASE / 'data' / 'raw' / 'ogbg_molhiv' / 'mapping' / 'mol.csv.gz')
labels = dataset.labels.flatten()

train_idx = set(split_idx['train'].tolist())
val_idx   = set(split_idx['valid'].tolist())
test_idx  = set(split_idx['test'].tolist())

df = smiles_df.copy()
df['y'] = labels
df['split'] = ['train' if i in train_idx else 'val' if i in val_idx else 'test' for i in range(len(df))]
print(f'loaded {len(df)} mols in {time.time()-t0:.1f}s')
print(df.groupby('split')['y'].agg(['count', 'sum', 'mean']).round(4))

loaded 41127 mols in 0.5s
       count   sum    mean
split                     
test    4113   130  0.0316
train  32901  1232  0.0374
val     4113    81  0.0197


In [3]:
# Compute Lipinski-14, RDKit Fragments-85, Morgan FP-1024, MACCS-167, Advanced-12 for every molecule.
# Mirrors Anthony's Phase 3 feature universe PLUS the new Fr_* family.

FRAG_FUNCS = [(n, getattr(Fragments, n)) for n in sorted(dir(Fragments)) if n.startswith('fr_')]
print('Fragment families:', len(FRAG_FUNCS))

def compute_all(smi):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    # Lipinski-14 (same as Anthony's data_pipeline)
    mw = Descriptors.MolWt(mol); logp = Descriptors.MolLogP(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol); hba = rdMolDescriptors.CalcNumHBA(mol)
    lip = {
        'mol_weight': mw, 'logp': logp, 'hbd': hbd, 'hba': hba,
        'tpsa': rdMolDescriptors.CalcTPSA(mol),
        'rotatable_bonds': rdMolDescriptors.CalcNumRotatableBonds(mol),
        'aromatic_rings': rdMolDescriptors.CalcNumAromaticRings(mol),
        'ring_count': rdMolDescriptors.CalcNumRings(mol),
        'heavy_atom_count': mol.GetNumHeavyAtoms(),
        'fraction_csp3': rdMolDescriptors.CalcFractionCSP3(mol),
        'molar_refractivity': Descriptors.MolMR(mol),
        'qed': Descriptors.qed(mol),
        'lipinski_violations': int(mw > 500) + int(logp > 5) + int(hbd > 5) + int(hba > 10),
        'passes_lipinski': int(mw <= 500 and logp <= 5 and hbd <= 5 and hba <= 10),
    }
    # Fragments-85 (MARK'S NEW CONTRIBUTION)
    frag = {n: f(mol) for n, f in FRAG_FUNCS}
    # Morgan 1024
    fp = np.array(AllChem.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=1024), dtype=np.int8)
    # MACCS-167
    maccs = np.array(MACCSkeys.GenMACCSKeys(mol), dtype=np.int8)
    # Advanced-12 (same as Anthony's list)
    adv = {
        'num_heteroatoms': rdMolDescriptors.CalcNumHeteroatoms(mol),
        'num_saturated_rings': rdMolDescriptors.CalcNumSaturatedRings(mol),
        'num_aliphatic_rings': rdMolDescriptors.CalcNumAliphaticRings(mol),
        'num_amide_bonds': rdMolDescriptors.CalcNumAmideBonds(mol),
        'num_spiro_atoms': rdMolDescriptors.CalcNumSpiroAtoms(mol),
        'num_bridgehead_atoms': rdMolDescriptors.CalcNumBridgeheadAtoms(mol),
        'labuteasa': Descriptors.LabuteASA(mol),
        'balabanj': Descriptors.BalabanJ(mol),
        'bertzct': Descriptors.BertzCT(mol),
        'maxpartialcharge': Descriptors.MaxPartialCharge(mol) or 0.0,
        'minpartialcharge': Descriptors.MinPartialCharge(mol) or 0.0,
        'numvalenceelectrons': Descriptors.NumValenceElectrons(mol),
    }
    return lip, frag, fp, maccs, adv

t0 = time.time()
lip_rows, frag_rows, fp_rows, maccs_rows, adv_rows, mask = [], [], [], [], [], []
for smi in df['smiles']:
    r = compute_all(smi)
    if r is None:
        mask.append(False); continue
    mask.append(True)
    l, fr, fp, mc, ad = r
    lip_rows.append(l); frag_rows.append(fr)
    fp_rows.append(fp); maccs_rows.append(mc); adv_rows.append(ad)
df_v = df[mask].reset_index(drop=True)
lip_df = pd.DataFrame(lip_rows)
frag_df = pd.DataFrame(frag_rows)
fp_arr = np.vstack(fp_rows)
maccs_arr = np.vstack(maccs_rows)
adv_df = pd.DataFrame(adv_rows)
print(f'featurized {len(df_v)} mols in {time.time()-t0:.1f}s')
print(f'Lipinski: {lip_df.shape}  Fragments: {frag_df.shape}  Morgan: {fp_arr.shape}  MACCS: {maccs_arr.shape}  Adv: {adv_df.shape}')

Fragment families: 85


[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerat

[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerat

[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerat

[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerator
[20:19:08] DEPRECATION WARNING: please use MorganGenerat

...

[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator


[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:24] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator


[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator


[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator
[20:26:25] DEPRECATION WARNING: please use MorganGenerator


featurized 41120 mols in 439.1s
Lipinski: (41120, 14)  Fragments: (41120, 85)  Morgan: (41120, 1024)  MACCS: (41120, 167)  Adv: (41120, 12)


In [4]:
# Assemble the feature matrices Anthony used + Mark's new fragment additions.
def slice_split(arr_or_df, split_name):
    idx = df_v['split'].values == split_name
    if isinstance(arr_or_df, pd.DataFrame):
        return arr_or_df[idx].values
    return arr_or_df[idx]
y = df_v['y'].values.astype(int)

# NaN-safe: some RDKit descriptors (BalabanJ, partial charges) can return NaN/inf on pathological mols
def clean(A):
    A = np.asarray(A, dtype=np.float32)
    A = np.nan_to_num(A, nan=0.0, posinf=1e6, neginf=-1e6)
    return A

lip_mat = clean(lip_df.values); frag_mat = clean(frag_df.values)
adv_mat = clean(adv_df.values); fp_mat = clean(fp_arr); maccs_mat = clean(maccs_arr)
print('post-clean NaN?', np.isnan(lip_mat).any(), np.isnan(adv_mat).any(), np.isnan(frag_mat).any())

FEATURE_SETS = {}
FEATURE_SETS['Lipinski (14)']            = lip_mat
FEATURE_SETS['Fragments (85)']           = frag_mat
FEATURE_SETS['Lip+Frag (99)']            = np.hstack([lip_mat, frag_mat])
FEATURE_SETS['Morgan (1024)']            = fp_mat
FEATURE_SETS['MACCS (167)']              = maccs_mat
FEATURE_SETS['AllTrad (1217)']           = np.hstack([lip_mat, fp_mat, maccs_mat, adv_mat])
FEATURE_SETS['AllTrad+Frag (1302)']      = np.hstack([lip_mat, frag_mat, fp_mat, maccs_mat, adv_mat])

# Build X_tr/X_va/X_te dict for each feature set
splits = {}
for name, X in FEATURE_SETS.items():
    splits[name] = {
        'X_tr': X[df_v['split'].values == 'train'],
        'X_va': X[df_v['split'].values == 'val'],
        'X_te': X[df_v['split'].values == 'test'],
    }
y_tr = y[df_v['split'].values == 'train']
y_va = y[df_v['split'].values == 'val']
y_te = y[df_v['split'].values == 'test']
print('splits built. train', y_tr.shape, 'val', y_va.shape, 'test', y_te.shape, 'pos rate tr', y_tr.mean().round(4))

post-clean NaN? False False False


splits built. train (32898,) val (4111,) test (4111,) pos rate tr 0.0374


In [5]:
# Experiment M3.1 — CatBoost head-to-head on 7 feature sets
# Same settings as Anthony's Phase 3 ablation (500 iter, depth=6, class_weights balanced)
CB_KW = dict(iterations=500, depth=6, learning_rate=0.05,
             loss_function='Logloss', auto_class_weights='Balanced',
             verbose=0, random_seed=42, allow_writing_files=False,
             thread_count=-1)

results = {}
for name, sp in splits.items():
    t0 = time.time()
    cb = CatBoostClassifier(**CB_KW)
    cb.fit(sp['X_tr'], y_tr, eval_set=(sp['X_va'], y_va))
    p_te = cb.predict_proba(sp['X_te'])[:, 1]
    auc = roc_auc_score(y_te, p_te); ap = average_precision_score(y_te, p_te)
    results[name] = {'test_auc': round(auc, 4), 'test_auprc': round(ap, 4),
                     'dims': sp['X_tr'].shape[1], 'fit_s': round(time.time()-t0, 1)}
    print(f"{name:30s}  dim={sp['X_tr'].shape[1]:5d}  AUC={auc:.4f}  AUPRC={ap:.4f}  ({time.time()-t0:.0f}s)")
pd.DataFrame(results).T.sort_values('test_auc', ascending=False)

Lipinski (14)                   dim=   14  AUC=0.7459  AUPRC=0.2758  (9s)


Fragments (85)                  dim=   85  AUC=0.6999  AUPRC=0.2411  (5s)


Lip+Frag (99)                   dim=   99  AUC=0.7247  AUPRC=0.2604  (8s)


Morgan (1024)                   dim= 1024  AUC=0.7448  AUPRC=0.3398  (47s)


MACCS (167)                     dim=  167  AUC=0.7670  AUPRC=0.3098  (20s)


AllTrad (1217)                  dim= 1217  AUC=0.7814  AUPRC=0.3490  (59s)


AllTrad+Frag (1302)             dim= 1302  AUC=0.7677  AUPRC=0.3647  (66s)


,test_auc,test_auprc,dims,fit_s
AllTrad (1217),0.7814,0.3490,1217.0,59.5
AllTrad+Frag (1302),0.7677,0.3647,1302.0,66.1
MACCS (167),0.7670,0.3098,167.0,19.8
Lipinski (14),0.7459,0.2758,14.0,9.3
Morgan (1024),0.7448,0.3398,1024.0,47.4
Lip+Frag (99),0.7247,0.2604,99.0,8.1
Fragments (85),0.6999,0.2411,85.0,4.7


### Checkpoint 1 — interpret Exp M3.1

Compare vs Anthony's Phase 3 CatBoost ablation:
- Anthony `Lipinski (14)` → 0.7744 · `AllTrad (1217)` → 0.7841 · `Full hybrid (1345)` → 0.7415

Did **Fragments (85)** alone clear Lipinski (0.7744)? If yes → H1 confirmed: explicit functional-group counts carry signal Lipinski flattens.
Did **AllTrad+Frag (1302)** beat `AllTrad (1217)`? If not → "more features still hurts" — the bottleneck is redundancy, which is exactly the setup for the feature-selection experiment next.

In [6]:
# Experiment M3.2 — Mutual-info feature selection sweep on AllTrad+Frag (1302 dims)
# Hypothesis: there's a sweet spot K << 1302 that beats both AllTrad-1217 and GIN+Edge-0.7860.
X_full_tr = splits['AllTrad+Frag (1302)']['X_tr']
X_full_va = splits['AllTrad+Frag (1302)']['X_va']
X_full_te = splits['AllTrad+Frag (1302)']['X_te']
print('computing mutual info on', X_full_tr.shape, '...')
t0 = time.time()
mi = mutual_info_classif(X_full_tr, y_tr, discrete_features='auto', random_state=42, n_neighbors=3)
print(f'done in {time.time()-t0:.0f}s, top5 MI =', np.sort(mi)[-5:].round(4))
rank = np.argsort(-mi)

K_grid = [20, 50, 100, 200, 400, 800, 1302]
sel_results = {}
for K in K_grid:
    idx = rank[:K]
    cb = CatBoostClassifier(**CB_KW)
    cb.fit(X_full_tr[:, idx], y_tr, eval_set=(X_full_va[:, idx], y_va))
    p = cb.predict_proba(X_full_te[:, idx])[:, 1]
    auc = roc_auc_score(y_te, p); ap = average_precision_score(y_te, p)
    sel_results[K] = {'auc': round(auc, 4), 'auprc': round(ap, 4)}
    print(f'K={K:4d}  AUC={auc:.4f}  AUPRC={ap:.4f}')
pd.DataFrame(sel_results).T

computing mutual info on (32898, 1302) ...


done in 309s, top5 MI = [0.0149 0.0168 0.0284 0.0314 0.0331]


K=  20  AUC=0.7702  AUPRC=0.2168


K=  50  AUC=0.7591  AUPRC=0.2793


K= 100  AUC=0.7892  AUPRC=0.3127


K= 200  AUC=0.8019  AUPRC=0.3285


K= 400  AUC=0.8105  AUPRC=0.3481


K= 800  AUC=0.7836  AUPRC=0.3421


K=1302  AUC=0.7673  AUPRC=0.3484


,auc,auprc
20,0.7702,0.2168
50,0.7591,0.2793
100,0.7892,0.3127
200,0.8019,0.3285
400,0.8105,0.3481
800,0.7836,0.3421
1302,0.7673,0.3484


In [7]:
# Combined comparison chart — Mark's results + Anthony's Phase 3 champions
ANTHONY = {
    'Ant: GIN+Edge (champion)': 0.7860,
    'Ant: AllTrad-1217':        0.7841,
    'Ant: Lipinski-14':         0.7744,
    'Ant: Full hybrid-1345':    0.7415,
    'Ant: GIN+VN':              0.7578,
}
best_sel_K = max(sel_results, key=lambda k: sel_results[k]['auc'])
MARK = {f'Mark: {n}': r['test_auc'] for n, r in results.items()}
MARK[f'Mark: MI-selected K={best_sel_K}'] = sel_results[best_sel_K]['auc']

combined = {**ANTHONY, **MARK}
order = sorted(combined.items(), key=lambda kv: kv[1])
names = [k for k, _ in order]; vals = [v for _, v in order]
colors = ['#1f77b4' if k.startswith('Ant') else '#ff7f0e' for k in names]

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(names, vals, color=colors)
ax.axvline(0.7860, color='red', ls='--', alpha=0.6, label="Anthony GIN+Edge champion")
for i, v in enumerate(vals):
    ax.text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=9)
ax.set_xlim(0.70, 0.80)
ax.set_xlabel('Test ROC-AUC (ogbg-molhiv)')
ax.set_title('Phase 3 leaderboard — Anthony (blue) vs Mark (orange)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig(RES / 'phase3_mark_leaderboard.png', dpi=120)
plt.close()
print('saved phase3_mark_leaderboard.png')

# Save JSON
out = {
    'phase': '3-mark',
    'date': '2026-04-08',
    'feature_set_ablation': results,
    'mi_selection_sweep': sel_results,
    'best_selected_K': int(best_sel_K),
    'anthony_champion_auc': 0.7860,
    'mark_best_auc': float(max([r['test_auc'] for r in results.values()] + [sel_results[best_sel_K]['auc']])),
}
(RES / 'phase3_mark_results.json').write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))

saved phase3_mark_leaderboard.png
{
  "phase": "3-mark",
  "date": "2026-04-08",
  "feature_set_ablation": {
    "Lipinski (14)": {
      "test_auc": 0.7459,
      "test_auprc": 0.2758,
      "dims": 14,
      "fit_s": 9.3
    },
    "Fragments (85)": {
      "test_auc": 0.6999,
      "test_auprc": 0.2411,
      "dims": 85,
      "fit_s": 4.7
    },
    "Lip+Frag (99)": {
      "test_auc": 0.7247,
      "test_auprc": 0.2604,
      "dims": 99,
      "fit_s": 8.1
    },
    "Morgan (1024)": {
      "test_auc": 0.7448,
      "test_auprc": 0.3398,
      "dims": 1024,
      "fit_s": 47.4
    },
    "MACCS (167)": {
      "test_auc": 0.767,
      "test_auprc": 0.3098,
      "dims": 167,
      "fit_s": 19.8
    },
    "AllTrad (1217)": {
      "test_auc": 0.7814,
      "test_auprc": 0.349,
      "dims": 1217,
      "fit_s": 59.5
    },
    "AllTrad+Frag (1302)": {
      "test_auc": 0.7677,
      "test_auprc": 0.3647,
      "dims": 1302,
      "fit_s": 66.1
    }
  },
  "mi_selection_sweep": {

### Iteration 1 — narrow the K sweet spot and audit what MI selected

First-pass result: **K=400 → 0.8105 AUC**, beating Anthony's GIN+Edge champion (0.7860) by **+0.0245**. The sweep looks clean (0.7702 → 0.8019 → 0.8105 → 0.7836 → 0.7673) — classic bias/variance bowl. Two follow-ups:

1. **Fine-grained K around 400** (300 / 350 / 400 / 450 / 500 / 600) — is 400 the true optimum or did I just happen to grid-land on it?
2. **Feature-category audit of K=400** — what fraction of the selected 400 come from Lipinski / Fragments / Morgan / MACCS / Adv? Does MI actually reach into Mark's new `Fr_*` family, or is the win driven entirely by Anthony's pre-existing feature pool?

Also noted: my `Lipinski (14)` pipeline reports 0.7459 vs Anthony's 0.7744. My matrices are float32-cleaned (NaN→0) which strips CatBoost's native missing-value handling. This shifts absolute numbers for small feature sets but leaves the K-sweep comparison self-consistent.

In [8]:
# Fine-grained K sweep + feature category audit
# AllTrad+Frag (1302) layout: [Lipinski 0..14) | Frag 14..99) | Morgan 99..1123) | MACCS 1123..1290) | Adv 1290..1302)
CAT_BOUNDS = [
    ('Lipinski', 0, 14),
    ('Fragments', 14, 99),
    ('Morgan',   99, 1123),
    ('MACCS',    1123, 1290),
    ('Advanced', 1290, 1302),
]
feature_category = np.empty(1302, dtype=object)
for name, a, b in CAT_BOUNDS:
    feature_category[a:b] = name

fine_K = [300, 350, 400, 450, 500, 600]
fine_results = {}
composition = {}
for K in fine_K:
    idx = rank[:K]
    cb = CatBoostClassifier(**CB_KW)
    cb.fit(X_full_tr[:, idx], y_tr, eval_set=(X_full_va[:, idx], y_va))
    p = cb.predict_proba(X_full_te[:, idx])[:, 1]
    auc = roc_auc_score(y_te, p); ap = average_precision_score(y_te, p)
    cats = pd.Series(feature_category[idx]).value_counts().to_dict()
    fine_results[K] = {'auc': round(auc, 4), 'auprc': round(ap, 4)}
    composition[K] = cats
    print(f"K={K:3d}  AUC={auc:.4f}  AUPRC={ap:.4f}  comp={cats}")

print('\n=== K=400 feature category composition ===')
k400 = composition[400]
total_pool = {n: (b - a) for n, a, b in CAT_BOUNDS}
for cat, pool in total_pool.items():
    chosen = k400.get(cat, 0)
    print(f"  {cat:10s}  selected {chosen:3d} / {pool:4d} pool  "
          f"({chosen / pool * 100:5.1f}% of category, {chosen / 400 * 100:5.1f}% of K=400)")

K=300  AUC=0.7883  AUPRC=0.3076  comp={'Morgan': 156, 'MACCS': 95, 'Fragments': 25, 'Lipinski': 14, 'Advanced': 10}


K=350  AUC=0.7813  AUPRC=0.3288  comp={'Morgan': 194, 'MACCS': 103, 'Fragments': 29, 'Lipinski': 14, 'Advanced': 10}


K=400  AUC=0.8105  AUPRC=0.3481  comp={'Morgan': 235, 'MACCS': 108, 'Fragments': 33, 'Lipinski': 14, 'Advanced': 10}


K=450  AUC=0.7796  AUPRC=0.3164  comp={'Morgan': 275, 'MACCS': 113, 'Fragments': 38, 'Lipinski': 14, 'Advanced': 10}


K=500  AUC=0.7945  AUPRC=0.3244  comp={'Morgan': 316, 'MACCS': 118, 'Fragments': 42, 'Lipinski': 14, 'Advanced': 10}


K=600  AUC=0.7941  AUPRC=0.3532  comp={'Morgan': 402, 'MACCS': 125, 'Fragments': 49, 'Lipinski': 14, 'Advanced': 10}

=== K=400 feature category composition ===
  Lipinski    selected  14 /   14 pool  (100.0% of category,   3.5% of K=400)
  Fragments   selected  33 /   85 pool  ( 38.8% of category,   8.2% of K=400)
  Morgan      selected 235 / 1024 pool  ( 22.9% of category,  58.8% of K=400)
  MACCS       selected 108 /  167 pool  ( 64.7% of category,  27.0% of K=400)
  Advanced    selected  10 /   12 pool  ( 83.3% of category,   2.5% of K=400)


In [9]:
# Final plot: K-sweep curve + category composition bar chart, save champion config
all_K_results = {**sel_results, **fine_results}
K_sorted = sorted(all_K_results.keys())
aucs = [all_K_results[k]['auc'] for k in K_sorted]
best_K = max(all_K_results, key=lambda k: all_K_results[k]['auc'])
best_auc = all_K_results[best_K]['auc']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(K_sorted, aucs, 'o-', color='#ff7f0e', lw=2, ms=8)
ax.axhline(0.7860, color='#1f77b4', ls='--', label='Anthony GIN+Edge 0.7860')
ax.axhline(0.7841, color='#2ca02c', ls=':', label='Anthony AllTrad-1217 0.7841')
ax.scatter([best_K], [best_auc], color='red', s=180, zorder=5,
           label=f'Mark champion K={best_K}: {best_auc:.4f}')
ax.set_xscale('log')
ax.set_xlabel('K (selected feature count, log scale)')
ax.set_ylabel('Test ROC-AUC (ogbg-molhiv)')
ax.set_title('Mutual-info feature-selection sweep')
ax.legend(loc='lower center')
ax.grid(alpha=0.3)

ax = axes[1]
cats = list(total_pool.keys())
pool_sz = [total_pool[c] for c in cats]
chosen_sz = [composition[best_K].get(c, 0) for c in cats]
rates = [cs / ps * 100 for cs, ps in zip(chosen_sz, pool_sz)]
x = np.arange(len(cats))
bars = ax.bar(x, rates, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
for i, (cs, ps) in enumerate(zip(chosen_sz, pool_sz)):
    ax.text(i, rates[i] + 1.5, f'{cs}/{ps}', ha='center', fontsize=10)
ax.set_xticks(x); ax.set_xticklabels(cats)
ax.set_ylabel('% of category selected')
ax.set_title(f'K={best_K} composition — MI-selected subset')
ax.set_ylim(0, max(rates) * 1.25 + 5)

plt.tight_layout()
plt.savefig(RES / 'phase3_mark_sweep_composition.png', dpi=120)
plt.close()
print('saved phase3_mark_sweep_composition.png')

# Update results JSON with iteration
out['fine_grained_sweep'] = fine_results
out['composition_by_K'] = {str(k): v for k, v in composition.items()}
out['champion_K'] = int(best_K)
out['champion_auc'] = float(best_auc)
out['mark_best_auc'] = float(best_auc)
out['delta_vs_anthony_gin_edge'] = round(best_auc - 0.7860, 4)
(RES / 'phase3_mark_results.json').write_text(json.dumps(out, indent=2))
print(f'\nCHAMPION: K={best_K}  AUC={best_auc:.4f}  vs Anthony GIN+Edge 0.7860 → Δ={best_auc - 0.7860:+.4f}')

saved phase3_mark_sweep_composition.png

CHAMPION: K=400  AUC=0.8105  vs Anthony GIN+Edge 0.7860 → Δ=+0.0245
